# CPSC 368 Project Phase 3

In [3]:
import pandas as pd

### 1. Unzip CSV Files and Read Data

In [ ]:
# Import data from zip files: *change folder path to data for zip files*
import zipfile

with zipfile.ZipFile("data/archive (8).zip", "r") as zip_ref:
    zip_ref.extractall("data")

with zipfile.ZipFile("data/archive (14).zip", "r") as zip_ref:
    zip_ref.extractall("data")
    
with zipfile.ZipFile("data/billboard.zip", "r") as zip_ref:
    zip_ref.extractall("data")

In [175]:
# Read data into df
charts_raw = pd.read_csv("data/charts.csv")
hot_audio_features_raw = pd.read_csv("data/Hot 100 Audio Features.csv")
hot_stuff_raw = pd.read_csv("data/Hot Stuff.csv")
spotify_songs_raw = pd.read_csv("data/spotify_songs.csv")

In [243]:
# View dataframes

#charts_raw.head()
#hot_audio_features_raw.head()
#hot_stuff_raw.head()
#spotify_songs_raw.head()

In [244]:
# Finding how many rows in original data
print(f"Dataframe 1: Spotify Charts has {len(charts_raw)} tuples")
print(f"Dataframe 2: Billboard Hot 100 Audio Features has {len(hot_audio_features_raw)} tuples")
print(f"Dataframe 3: Billboard Hot Stuff has {len(hot_stuff_raw)} tuples")
print(f"Dataframe 4: Spotify Songs has {len(spotify_songs_raw)} tuples")

Dataframe 1: Spotify Charts has 26173514 tuples
Dataframe 2: Billboard Hot 100 Audio Features has 29503 tuples
Dataframe 3: Billboard Hot Stuff has 327895 tuples
Dataframe 4: Spotify Songs has 32833 tuples


### 2. Preliminary Cleaning: Dropping Columns

In [ ]:
# use hot_stuff: charting weeks
# use hot audio features: for spotify track ID
# use charts: top 100 in CA
# use spotify_songs: audio features 

# 1. spotify charts 
# drop all other columns. title, artist, date, region, url (used to extract track_id), rank, streams
charts = charts_raw[["title", "artist", "date", "region", "url", "rank", "streams", "chart"]]

# 2. billboard hot stuff
# drop all other columns. SongID (Join Key), WeekID (to be standardized as Date), Week Position (rank), Weeks on Chart, Instance.
hot_stuff = hot_stuff_raw[["Song", "Performer", "SongID",
                           "WeekID", "Week Position", "Weeks on Chart", "Instance"]]

# 3. billboard hot 100 audio features 
hot_audio_features = hot_audio_features_raw[["SongID", "spotify_track_id"]]

# 4. spotify_songs 
# drop all other columns. track_id (Primary Key), track_name, track_artist, danceability, speechiness, track_popularity, track_album_release_date

spotify_songs = spotify_songs_raw[["track_id", "track_name", "track_artist", "track_popularity", "track_album_release_date", "danceability", "speechiness"]]

### 3. Cleaning Datasets

##### Part 1: Spotify Songs

In [ ]:
# remove duplicates and tuples with NA track id
spotify_songs = spotify_songs.drop_duplicates().dropna(subset=["track_id"])
spotify_ids = set(spotify_songs["track_id"])

In [308]:
len(spotify_songs)

28356

##### Billboard Dataset

In [ ]:
# step 1: joining billboard datasets for spotify_track_id
billboard_weeks = pd.merge(
  hot_audio_features,
  hot_stuff,
  on="SongID", 
  how="inner"
).dropna(subset=["spotify_track_id"])

# step 2: rename columns
billboard_weeks = billboard_weeks.rename(columns={
  "Song": "song_name",
  "Performer":"artist_name", 
  "WeekID": "date", 
  "Week Position":"week_position",
  "Weeks on Chart":"weeks_on_chart",
  "Instance": "instance" 
}
).drop(columns=["SongID"])

# step 3: standardize date
billboard_weeks["date"] = pd.to_datetime(billboard_weeks["date"], format="%m/%d/%Y")

# step 4: filter for only year 2021 
billboard_weeks = billboard_weeks[
  (billboard_weeks["date"] >= "2019-01-01") &
  (billboard_weeks["date"] <= "2019-12-31") 
]

# # step 5: reducing number of rows. aggregating to taking max # weeks on chart per instance and track id 
billboard_weeks_max = (
    billboard_weeks
    .groupby(["spotify_track_id", "instance"], as_index=False)
    .agg({
        "song_name": "first",
        "artist_name": "first",
        "date": "max",
        "week_position": "min",
        "weeks_on_chart": "max"
    })
     .rename(columns={
        "date": "end_date",
        "week_position": "best_week_position",
        "weeks_on_chart": "weeks_on_chart_run"
    })
)

# step 6: only keep tuples where track_id is in spotify_songs
billboard_weeks_max = billboard_weeks_max[
  billboard_weeks_max["spotify_track_id"].isin(spotify_ids)
]


In [372]:
len(billboard_weeks_max)

297

##### Spotify Charts

In [ ]:
# main approach: take top 5 songs everyday of 2021

# step 1: standardize date column 
charts["date"] = pd.to_datetime(charts["date"], format="%Y-%m-%d")

# step 2: filter for year 2021, region in Canada and chart type "top200" and only top 5 songs everyday
charts = charts[
  (charts["date"] >= "2019-01-01") &
  (charts["date"] <= "2019-12-31") &
  (charts["region"]=="Canada") & 
  (charts["chart"]=="top200") 
]
# step 3: drop any NA urls and extract track id from url
charts = charts.dropna(subset=["url"])
charts["extracted_track_id"] = charts["url"].str.split("/track/").str[1]

# step 4: only keep tuples where track_id is in spotify_songs
charts = charts[
  charts["extracted_track_id"].isin(spotify_ids)
]

# step 5: pick random date every month to reduce number of tuples
charts["month"] = charts["date"].dt.to_period("M")

random_days = (
    charts[["month", "date"]]
    .drop_duplicates()
    .groupby("month")
    .sample(1, random_state=1)
)

charts = charts.merge(random_days, on=["month", "date"])

In [365]:
len(charts)

1701

In [ ]:
# approach b: if aggregation at this part is allowed, group by trackid and sum # streams for all of 2021. 
# charts_grouped = (
#   charts
#   .groupby("extracted_track_id", as_index=False)
#   .agg({
#     "title": "first",
#     "artist": "first", 
#     "streams": "sum"
#   })
#   .rename(columns={"streams":"total_streams"})
# )

# len(charts_grouped)

710

#### Part 2: Spotify Songs

In [373]:
# requirement: 2000 tuples have all songs from other dataframes (primary key is track_id in spotify_songs) 

# step 1: find all track_ids in other dataframes  
billboard_ids = set(billboard_weeks_max["spotify_track_id"])
charts_ids = set(charts["extracted_track_id"])
required_ids = billboard_ids.union(charts_ids)

# step 2: make sure all required songs are in dataframe 
spotify_required = spotify_songs[spotify_songs["track_id"].isin(required_ids)]

# step 3: find songs that are in the original dataset that aren't required
spotify_extra = spotify_songs[~spotify_songs["track_id"].isin(required_ids)]

# step 4: find number of tuples that can be filled by non-required songs
spotify_extra = spotify_extra.head(2000 - len(spotify_required))

# step 5: combine the datasets to 2000 tuples
spotify_songs_final = pd.concat([spotify_required, spotify_extra]).drop_duplicates(subset=["track_id"])


In [374]:
len(spotify_songs_final)

2000

### 4. Export Data to CSV

In [377]:
charts.to_csv("cleaned_data/charts.csv", index=False)
billboard_weeks_max.to_csv("cleaned_data/billboard_weeks.csv", index=False)
spotify_songs_final.to_csv("cleaned_data/spotify_songs.csv", index=False)